In [1]:
import apache_beam as beam
import json
import importlib
import os
import unittest
import pandas as pd
import numpy as np

from apache_beam.transforms.util import WaitOn
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime

Failed to import GCSFileSystem; loading of this filesystem will be skipped. Error details: cannot import name 'storage' from 'google.cloud' (unknown location)


In [2]:
fromNotebook = True
configFile = "warehouse_wwi.json"
loadConfigFile = "../load/load_wwi.json"
newCutoffDate = "2013-01-01"
dimension_tables = []
fact_tables = []
archivePath = os.getcwd()
is_rollback = False
process_id = 1
no_of_workers = 1
table_type = ""
release_github_repo = ""
release_github_branch = ""
release_github_tag = ""
modules_directory = "../modules"

In [3]:
spec = importlib.util.spec_from_file_location("sql_utils", f"{modules_directory}/sql_utilities.py")
sql_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sql_utils)

In [4]:
f = open(loadConfigFile,)
source_config = json.load(f)
f.close()

load_tables = [
    {
        "name": table["name"],
        "database": source_config["destination"]["database"],
        "schema": table["destination"]["schema"],
        "table": table["destination"]["table"]
    }
    for table in source_config["tables"]
]

print(f"load_tables: {load_tables}")

load_tables: [{'name': 'ApplicationCities', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_Cities'}, {'name': 'ApplicationCitiesArchive', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_Cities_Archive'}, {'name': 'ApplicationCountries', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_Countries'}, {'name': 'ApplicationCountriesArchive', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_Countries_Archive'}, {'name': 'ApplicationDeliveryMethods', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_DeliveryMethods'}, {'name': 'ApplicationDeliveryMethodsArchive', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_DeliveryMethods_Archive'}, {'name': 'ApplicationPaymentMethods', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_PaymentMethods'}, {'name': 'ApplicationPaymentMethodsArchive', 'database': 'WideWorldImpor

In [6]:
f = open(configFile,)
config = json.load(f)
f.close()

source_db = config["source"]
destination_db = config["destination"]
initialCutoffDate = config["initialCutoffDate"]
if fromNotebook == True:
    dimension_tables = config["dimensionTables"]
    fact_tables = config["factTables"]
    newCutoffDate = config["cutoffDate"]
    is_rollback = config["isRollback"]
else:
    if table_type == "" or table_type == "dimension":
        dimension_table_groups = np.array_split(config["dimensionTables"], no_of_workers)

        if len(dimension_table_groups) >= process_id:
            dimension_tables = list(dimension_table_groups[process_id - 1])

    if table_type == "" or table_type == "fact":
        fact_table_groups = np.array_split(config["factTables"], no_of_workers)

        if len(fact_table_groups) >= process_id:
            fact_tables = list(fact_table_groups[process_id - 1])

lastCutoffDate = "2012-12-31"
warehouse_tables = (
    [
        {
            "name": table["name"],
            "database": config["destination"]["database"],
            "schema": table["schema"],
            "table": table["table"]
        }
        for table in config["dimensionTables"]
    ] + 
    [
        {
            "name": table["name"],
            "database": config["destination"]["database"],
            "schema": table["schema"],
            "table": table["table"]
        }
        for table in config["factTables"]
    ]
)
spark_master = config["sparkMaster"]
spark_jars = config["sparkJars"]
spark_executor_memory = config["sparkExecutorMemory"]
spark_driver_memory = config["sparkDriverMemory"]
spark_executor_cores = config["sparkExecutorCores"]
spark_cores_max = config["sparkCoresMax"]
spark_network_timeout = config["sparkNetworkTimeout"]
spark_executor_heartbeat_interval = config["sparkExecutorHeartBeatInterval"]
runner_options = config["runnerOptions"][config["runner"]]
gcp_credential = config["gcpCredential"]
script_directory = f"{archivePath}/"

print(f"source_db: {source_db}")
print(f"destination_db: {destination_db}")
print(f"dimension_tables: {dimension_tables}")
print(f"fact_tables: {fact_tables}")
print(f"initialCutoffDate: {initialCutoffDate}")
print(f"newCutoffDate: {newCutoffDate}")
print(f"lastCutoffDate: {lastCutoffDate}")
print(f"warehouse_tables: {warehouse_tables}")
print(f"spark_master: {spark_master}")
print(f"spark_jars: {spark_jars}")
print(f"spark_executor_memory: {spark_executor_memory}")
print(f"spark_driver_memory: {spark_driver_memory}")
print(f"spark_executor_cores: {spark_executor_cores}")
print(f"spark_cores_max: {spark_cores_max}")
print(f"spark_network_timeout: {spark_network_timeout}")
print(f"spark_executor_heartbeat_interval: {spark_executor_heartbeat_interval}")
print(f"runner_options: {runner_options}")
print(f"gcp_credential: {gcp_credential}")
print(f"is_rollback: {is_rollback}")
print(f"process_id: {process_id}")
print(f"no_of_workers: {no_of_workers}")
print(f"script_directory: {script_directory}")

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = gcp_credential

source_db: {'databaseType': 'mssql', 'database': 'WideWorldImportersDW', 'conn': 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost,1433;DATABASE=WideWorldImportersDW;UID=sa;PWD=PP@$$w0rd', 'conn_spark': 'jdbc:sqlserver://localhost;database=WideWorldImportersDW;user=sa;password=PP@$$w0rd;encrypt=false'}
destination_db: {'databaseType': 'mssql', 'database': 'WideWorldImportersDW', 'conn': 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost,1433;DATABASE=WideWorldImportersDW;UID=sa;PWD=PP@$$w0rd', 'conn_spark': 'jdbc:sqlserver://localhost;database=WideWorldImportersDW;user=sa;password=PP@$$w0rd;encrypt=false'}
dimension_tables: [{'rollbackVersion': 1, 'name': 'DimCities', 'schema': 'dbo', 'table': 'DimCities', 'createTable': 'scripts/warehouse_wwi_mssql/create_dimension_cities.sql', 'upsertTable': 'scripts/warehouse_wwi_mssql/upsert_dimension_cities_2013-01-01.sql', 'specificColumnTypes': [{'name': 'Location', 'type': 'VARBINARY(MAX)', 'typeCast': 'CAST(? AS VARBINARY(MAX))'}]

In [7]:
if source_db["databaseType"].startswith("spark-") and destination_db["databaseType"].startswith("spark-"):
    sql_utils.initialize_spark_session(
        master=spark_master,
        jdbc_url=source_db["conn"], 
        jars=spark_jars, 
        executor_memory=spark_executor_memory, 
        driver_memory=spark_driver_memory, 
        executor_cores=spark_executor_cores, 
        cores_max=spark_cores_max, 
        network_timeout=spark_network_timeout, 
        executor_heartbeat_interval=spark_executor_heartbeat_interval
    )

In [8]:
def get_last_cutoff_date(table, schema, last_cutoff_dates):
    last_cutoff_date = next((item for item in last_cutoff_dates if item.get("TableName") == table and item.get("SchemaName") == schema), None)

    if last_cutoff_date is None:
        return initialCutoffDate
    else:
        return last_cutoff_date["LastCutoffDate"].strftime("%Y-%m-%d %H:%M:%S")

In [9]:
def upsert_data_base(conn_str, database_type, database, upsert_path, values_to_replace, tables_to_replace, show_sql = False):
    sql = sql_utils.get_sql_from_script(
        path=upsert_path, 
        values=values_to_replace, 
        tables=tables_to_replace,
        database_type=database_type,
        directory=script_directory
    )

    if show_sql == True:
        print(f"""Upsert Data sql:
                {sql} 
                """)
    
    sql_utils.exec_sql(conn_str, sql, None, database_type, database, spark_master, spark_jars)

In [10]:
class UpsertData(beam.DoFn):
    def __init__(self, database_type, conn_str, upsert_path, values_to_replace, tables_to_replace, yield_record = None, show_sql = False, database = ""):
        self.database_type = database_type
        self.conn_str = conn_str
        self.upsert_path = upsert_path
        self.values_to_replace = values_to_replace
        self.tables_to_replace = tables_to_replace
        self.yield_record = yield_record
        self.show_sql = show_sql
        self.database = database

    def process(self, element):
        upsert_data_base(
            self.conn_str, 
            self.database_type, 
            self.database, 
            self.upsert_path, 
            self.values_to_replace, 
            self.tables_to_replace, 
            self.show_sql
        )

        if self.yield_record is not None:
            yield self.yield_record

In [11]:
pd.set_option('display.max_colwidth', None)
tc = unittest.TestCase()

class WarehouseData(beam.DoFn):
    def __init__(self, upsert_config = None, assert_config = None, insert_history_config = None, modify_table_config = None, 
        yield_record = None, rollback_droprecords_config = None, rollback_drop_table_config = None, rollback_recreate_table_config = None,
        rollback_final_config = None):
        self.upsert_config = upsert_config
        self.assert_config = assert_config
        self.insert_history_config = insert_history_config
        self.modify_table_config = modify_table_config
        self.yield_record = yield_record
        self.rollback_droprecords_config = rollback_droprecords_config
        self.rollback_drop_table_config = rollback_drop_table_config
        self.rollback_recreate_table_config = rollback_recreate_table_config
        self.rollback_final_config = rollback_final_config
    
    def upsert_data(self):
        upsert_data_base(
            self.upsert_config["conn"],
            self.upsert_config["database_type"], 
            self.upsert_config["database"], 
            self.upsert_config["path"], 
            self.upsert_config["values_to_replace"], 
            self.upsert_config["tables_to_replace"], 
            self.upsert_config["show_sql"]
        )

    def assert_data(self):
        if self.assert_config["test"] == True:
            error_counts = []

            if len(self.assert_config["unit_tests"]) > 0:
                count_sqls = []
                
                for unit_test in self.assert_config["unit_tests"]:
                    common_values = [
                        { "name": "Name", "value": unit_test["name"] },
                        { "name": "Column", "value": unit_test["column"] },
                        { "name": "Type", "value": unit_test["type"] },
                        { "name": "SubType", "value": unit_test["subType"] }
                    ]

                    if unit_test["type"] == "column":
                        if unit_test["subType"] == "relationship":
                            count_sqls.append(
                                sql_utils.get_sql_from_script(
                                    path=self.assert_config["test_config"][unit_test["subType"]],
                                    values=self.assert_config["values_to_replace"] + common_values + [
                                        { "name": "ParentColumn", "value": unit_test["parentColumn"] },
                                        { "name": "ParentTable", "value": unit_test["parentTable"] }
                                    ], 
                                    tables=self.assert_config["tables_to_replace"],
                                    database_type=self.assert_config["database_type"],
                                    directory=script_directory
                                ) 
                            )
                        elif unit_test["subType"] == "expressionTrue":
                            count_sqls.append(
                                sql_utils.get_sql_from_script(
                                    path=self.assert_config["test_config"][unit_test["subType"]],
                                    values=self.assert_config["values_to_replace"] + common_values + [
                                        { "name": "Expression", "value": unit_test["expression"] }
                                    ], 
                                    tables=self.assert_config["tables_to_replace"],
                                    database_type=self.assert_config["database_type"],
                                    directory=script_directory
                                ) 
                            )
                        else:
                            count_sqls.append(
                                sql_utils.get_sql_from_script(
                                    path=self.assert_config["test_config"][unit_test["subType"]],
                                    values=self.assert_config["values_to_replace"] + common_values, 
                                    tables=self.assert_config["tables_to_replace"],
                                    database_type=self.assert_config["database_type"],
                                    directory=script_directory
                                ) 
                            )
                    elif unit_test["type"] == "table":
                        count_sqls.append(
                            sql_utils.get_sql_from_script(
                                path=unit_test["testScript"],
                                values=self.assert_config["values_to_replace"] + common_values, 
                                tables=self.assert_config["tables_to_replace"],
                                database_type=self.assert_config["database_type"],
                                directory=script_directory
                            ) 
                        )

                if len(count_sqls) > 0:
                    count_sql = sql_utils.get_sql_from_script(
                        path=self.assert_config["test_config"]["selectCounts"],
                        values=[
                            {
                                "name": "SelectCountsSql",
                                "value": " UNION ALL ".join(count_sqls)
                            }
                        ],
                        tables=self.assert_config["tables_to_replace"],
                        database_type=self.assert_config["database_type"],
                        directory=script_directory
                    )

                    if self.assert_config["show_sql"] == True:
                        print(f"count_sql: {count_sql}",)

                    error_counts = list(
                        sql_utils.select_sql(
                            self.assert_config["conn"], 
                            count_sql, 
                            self.assert_config["database_type"],
                            "dictionary",
                            self.assert_config["database"],
                            spark_jars,
                            spark_master,
                            False
                        )
                    )
                    
            if len(error_counts) > 0:
                print("Error occurred on the following tables:")
                
                display(pd.DataFrame(error_counts))

                raise Exception("Assert data error.  Please see error above.")
            
    def insert_history_record(self):
        upsert_data_base(
            self.insert_history_config["conn"],
            self.insert_history_config["database_type"], 
            self.insert_history_config["database"], 
            self.insert_history_config["path"], 
            self.insert_history_config["values_to_replace"], 
            self.insert_history_config["tables_to_replace"], 
            self.insert_history_config["show_sql"]
        )

    def modify_table(self):
        upsert_data_base(
            self.modify_table_config["conn"],
            self.modify_table_config["database_type"], 
            self.modify_table_config["database"], 
            self.modify_table_config["path"], 
            self.modify_table_config["values_to_replace"], 
            self.modify_table_config["tables_to_replace"], 
            self.modify_table_config["show_sql"]
        )

    def rollback_droprecords(self):
        added_columns_sql = []

        if len(self.rollback_droprecords_config["added_columns"]) > 0:
            for column in self.rollback_droprecords_config["added_columns"].split(","):
                added_columns_sql.append(
                    sql_utils.get_sql_from_script(
                        path=self.rollback_droprecords_config["drop_column_path"], 
                        values=[
                            { "name": "Schema", "value": self.rollback_droprecords_config["schema"] },
                            { "name": "Table", "value": self.rollback_droprecords_config["table"] },
                            { "name": "Database", "value": destination_db["database"] },
                            { "name": "Column", "value": column }
                        ], 
                        tables=[],
                        directory=script_directory
                    )
                )

        upsert_data_base(
            self.rollback_droprecords_config["conn"],
            self.rollback_droprecords_config["database_type"], 
            self.rollback_droprecords_config["database"], 
            self.rollback_droprecords_config["path"], 
            self.rollback_droprecords_config["values_to_replace"] + [{ "name": "DropColumns", "value": " ".join(added_columns_sql) }], 
            self.rollback_droprecords_config["tables_to_replace"], 
            self.rollback_droprecords_config["show_sql"]
        )

    def rollback_drop_table(self):
        upsert_data_base(
            self.rollback_drop_table_config["conn"],
            self.rollback_drop_table_config["database_type"], 
            self.rollback_drop_table_config["database"], 
            self.rollback_drop_table_config["path"], 
            self.rollback_drop_table_config["values_to_replace"], 
            self.rollback_drop_table_config["tables_to_replace"], 
            self.rollback_drop_table_config["show_sql"]
        )

    def rollback_recreate_table(self):
        upsert_data_base(
            self.rollback_recreate_table_config["conn"],
            self.rollback_recreate_table_config["database_type"], 
            self.rollback_recreate_table_config["database"], 
            self.rollback_recreate_table_config["path"], 
            self.rollback_recreate_table_config["values_to_replace"], 
            self.rollback_recreate_table_config["tables_to_replace"], 
            self.rollback_recreate_table_config["show_sql"]
        )

    def rollback_recreate_table(self):
        upsert_data_base(
            self.rollback_recreate_table_config["conn"],
            self.rollback_recreate_table_config["database_type"], 
            self.rollback_recreate_table_config["database"], 
            self.rollback_recreate_table_config["path"], 
            self.rollback_recreate_table_config["values_to_replace"], 
            self.rollback_recreate_table_config["tables_to_replace"], 
            self.rollback_recreate_table_config["show_sql"]
        )

    def rollback_final(self):
        upsert_data_base(
            self.rollback_final_config["conn"],
            self.rollback_final_config["database_type"], 
            self.rollback_final_config["database"], 
            self.rollback_final_config["path"], 
            self.rollback_final_config["values_to_replace"], 
            self.rollback_final_config["tables_to_replace"], 
            self.rollback_final_config["show_sql"]
        )

    def process(self, element):
        # Modify Table
        if self.modify_table_config is not None:
            self.modify_table()

        # Rollback Drop Records
        if self.rollback_droprecords_config is not None:
            self.rollback_droprecords()

        # Drop Table for Rollback
        if self.rollback_drop_table_config is not None:
            self.rollback_drop_table()

        # Recreate Table for Rollback
        if self.rollback_recreate_table_config is not None:
            self.rollback_recreate_table()

        # Update Histories for Rollback
        if self.rollback_final_config is not None:
            self.rollback_final()

        # Upsert Data
        if self.upsert_config is not None:
            self.upsert_data()

        # Assert Data
        if self.assert_config is not None:
            self.assert_data()

        # Insert History Record
        if self.insert_history_config is not None:
            self.insert_history_record()

        if self.yield_record is not None:
            yield self.yield_record

In [12]:
class LoadData(beam.DoFn):
    def __init__(self, database_type, conn_str, load_path, values_to_replace, tables_to_replace, show_sql = False, result_type = "tuple", set_timestamp_tostring = False):
        self.database_type = database_type
        self.conn_str = conn_str
        self.load_path = load_path
        self.values_to_replace = values_to_replace
        self.tables_to_replace = tables_to_replace
        self.show_sql = show_sql
        self.result_type = result_type
        self.set_timestamp_tostring = set_timestamp_tostring

    def process(self, element):
        sql = sql_utils.get_sql_from_script(
            path=self.load_path, 
            values=self.values_to_replace, 
            tables=self.tables_to_replace,
            directory=script_directory
        )

        if self.show_sql == True:
            print(f"LoadData sql: {sql}")
        
        return sql_utils.select_sql(self.conn_str, sql, self.database_type, self.result_type, "", spark_jars, spark_master, self.set_timestamp_tostring)

In [38]:
class CreateTables(beam.DoFn):
    def __init__(self, database_type, conn_str, sql, database, yield_record = None, show_sql = False):
        self.database_type = database_type
        self.conn_str = conn_str
        self.sql = sql
        self.database = database
        self.yield_record = yield_record
        self.show_sql = show_sql

    def process(self, element):
        if len(self.sql) > 0:
            
            if self.show_sql == True:
                print(f"""Upsert Data sql:
                    {self.sql} 
                    """)
                
                print(f"element: {element}")
                print(f"len element: {len(element)}")

            sql_utils.exec_sql(self.conn_str, self.sql, None, self.database_type, self.database, spark_master, spark_jars)

        if self.yield_record is not None:
            yield self.yield_record

In [39]:
def get_last_cutoff_dates():
    sql = sql_utils.get_sql_from_script(
        path=config["warehouseHistory"]["loadLastCutoffDatesTable"], 
        values=[
            { "name": "Schema", "value": config["warehouseHistory"]["schema"] },
            { "name": "Table", "value": config["warehouseHistory"]["table"] },
            { "name": "Database", "value": destination_db["database"] }
        ], 
        tables=[],
        directory=script_directory
    )

    # print(f"sql: {sql}")

    return sql_utils.select_sql(
        destination_db["conn"], 
        sql, 
        destination_db["databaseType"], 
        "dictionary", 
        destination_db["database"], 
        spark_jars, 
        spark_master, 
        False, 
        False,
        spark_load_sql = True
    )

In [40]:
def get_create_tables_sql(tables, last_cutoff_dates, new_cutoff_date):
    create_table_sqls = []

    for table in tables:
        tableName = table["table"]
        last_cutoff_date = get_last_cutoff_date(tableName, table["schema"], last_cutoff_dates)
        lastCutoffDateVal = datetime.strptime(last_cutoff_date, "%Y-%m-%d %H:%M:%S")
        newCutoffDateVal = datetime.strptime(new_cutoff_date, "%Y-%m-%d %H:%M:%S")

        if lastCutoffDateVal < newCutoffDateVal:
            create_table_sqls.append(
                sql_utils.get_sql_from_script(
                    path=table["createTable"], 
                    values=[
                        { "name": "Database", "value": destination_db["database"] },
                        { "name": "Schema", "value": table["schema"] },
                        { "name": "Table", "value": table["table"] }
                    ], 
                    tables=[],
                    directory=script_directory
                )
                
            )

    if len(create_table_sqls) > 0:
        return " ".join(create_table_sqls)
    
    return ""

In [41]:
if is_rollback == False:
    last_cutoff_dates = get_last_cutoff_dates()
    print(f"last_cutoff_dates: {last_cutoff_dates}")

    options = PipelineOptions(runner_options)

    with beam.Pipeline(options=options) as p:
        def modify_table(table, process_table, last_cutoff_dates, wait_on_process):
            tableName = table["table"]
            last_cutoff_date = get_last_cutoff_date(tableName, table["schema"], last_cutoff_dates)
            modifyTableScripts = []
                    
            for modifyTableScript in table["modifyTable"]:
                modifyTableScripts.append(
                    sql_utils.get_sql_from_script(
                        path=config["modifyWarehouseHistory"]["loadUnprocessedModifyWarehouseHistoryScriptTable"],
                        values=[
                            { "name": "ScriptName", "value": modifyTableScript["tableUpdate"] },
                            { "name": "LoadScriptName", "value": modifyTableScript["tableDataLoad"] },
                            { "name": "UpdateScriptName", "value": modifyTableScript["tableDataUpdate"] },
                            { "name": "Name", "value": modifyTableScript["name"] }
                        ],
                        tables=[],
                        database_type=destination_db["databaseType"],
                        directory=script_directory
                    )
                )
            
            unprocessedScripts = sql_utils.select_spark_sql(
                destination_db["conn"],
                sql_utils.get_sql_from_script(
                    path=config["modifyWarehouseHistory"]["loadUnprocessedModifyWarehouseHistoryTable"],
                    values=[
                        { "name": "Database", "value": destination_db["database"] },
                        { "name": "Schema", "value": config["modifyWarehouseHistory"]["schema"] },
                        { "name": "Table", "value": config["modifyWarehouseHistory"]["table"] },
                        { "name": "TableName", "value": tableName },
                        { "name": "SchemaName", "value": table["schema"] },
                        { "name": "LoadDate", "value": last_cutoff_date },
                        { "name": "MHSchema", "value": config["warehouseHistory"]["schema"] },
                        { "name": "MHTable", "value": config["warehouseHistory"]["table"] },
                        { "name": "ModifyTableScripts", "value": " UNION ".join(modifyTableScripts) }
                    ],
                    tables=[],
                    database_type=destination_db["databaseType"],
                    directory=script_directory
                ),
                destination_db["databaseType"],
                destination_db["database"],
                spark_master,
                spark_jars
            )

            for script in unprocessedScripts:
                print(f"Processing '{script["Name"]}' for table {tableName} for cutoff {newCutoffDate}.")
                
                sql_utils.exec_sql(
                    destination_db["conn"],
                    sql_utils.get_sql_from_script(
                        path=script["ScriptName"], 
                        values=[
                            { "name": "Database", "value": destination_db["database"] },
                            { "name": "Schema", "value": table["schema"] },
                            { "name": "Table", "value": table["table"] },
                        ],
                        tables=[],
                        database_type=destination_db["databaseType"],
                        directory=script_directory
                    ),
                    None,
                    destination_db["databaseType"],
                    destination_db["database"]
                )

                upsert_config = None
                if script["UpdateScriptName"] != "":
                    upsert_config = {
                        "conn": destination_db["conn"],
                        "database_type": destination_db["databaseType"],
                        "database": destination_db["database"],
                        "path": script["UpdateScriptName"],
                        "values_to_replace": [
                            { "name": "NewCutoffDate", "value": newCutoffDate },
                            { "name": "LastCutoffDate", "value": last_cutoff_date }
                        ],
                        "tables_to_replace": load_tables + warehouse_tables,
                        "show_sql": False
                    }

                process_table = (
                    process_table
                    | f"Modify Table {tableName}" >> 
                        beam.ParDo(
                            WarehouseData(
                                modify_table_config={
                                    "conn": destination_db["conn"],
                                    "database_type": destination_db["databaseType"],
                                    "database": destination_db["database"],
                                    "path": script["ScriptName"],
                                    "values_to_replace": [
                                        { "name": "Database", "value": destination_db["database"] },
                                        { "name": "Schema", "value": table["schema"] },
                                        { "name": "Table", "value": table["table"] },
                                    ],
                                    "tables_to_replace": [],
                                    "show_sql": False
                                },
                                upsert_config=upsert_config,
                                assert_config={
                                    "conn": destination_db["conn"],
                                    "database_type": destination_db["databaseType"],
                                    "database": destination_db["database"],
                                    "test_config": config["testConfig"],
                                    "unit_tests": modifyTableScript["unitTests"],
                                    "values_to_replace": [
                                        { "name": "Database", "value": destination_db["database"] },
                                        { "name": "Schema", "value": table["schema"] },
                                        { "name": "Table", "value": table["table"] },
                                        { "name": "LoadDate", "value": last_cutoff_date },
                                        { "name": "LastCutoffDate", "value": last_cutoff_date },
                                        { "name": "NewCutoffDate", "value": newCutoffDate }
                                    ],
                                    "tables_to_replace": load_tables + warehouse_tables,
                                    "test": config["test"],
                                    "test_record_count": config["testRecordCount"],
                                    "show_sql": False
                                },
                                insert_history_config={
                                    "conn": destination_db["conn"],
                                    "database_type": destination_db["databaseType"],
                                    "database": destination_db["database"],
                                    "path": config["modifyWarehouseHistory"]["insertModifyWarehouseHistoryTable"],
                                    "values_to_replace": [
                                        { "name": "Database", "value": destination_db["database"] },
                                        { "name": "Schema", "value": config["modifyWarehouseHistory"]["schema"] },
                                        { "name": "Table", "value": config["modifyWarehouseHistory"]["table"] },
                                        { "name": "TableName", "value": tableName },
                                        { "name": "SchemaName", "value":  table["schema"] },
                                        { "name": "LoadDate", "value": last_cutoff_date },
                                        { "name": "Status", "value": "Successful" },
                                        { "name": "Details", "value": "" },
                                        { "name": "ScriptName", "value": script["Name"] }
                                    ],
                                    "tables_to_replace": [],
                                    "show_sql": False
                                },
                                yield_record={ f"Successfully processed table {tableName} for {newCutoffDate}." }
                            )
                        )
                    | f"Modify WaitOn Table {tableName} 4" >> WaitOn(*wait_on_process)
                )
            
            return process_table

        def get_process_table(table, wait_on_process, last_cutoff_dates):
            tableName = table["name"]

            print(f"Processing table {tableName} for {newCutoffDate}.")

            last_cutoff_date = get_last_cutoff_date(tableName, table["schema"], last_cutoff_dates)
            lastCutoffDateVal = datetime.strptime(last_cutoff_date, "%Y-%m-%d %H:%M:%S")
            newCutoffDateVal = datetime.strptime(newCutoffDate, "%Y-%m-%d %H:%M:%S")
            
            if lastCutoffDateVal >= newCutoffDateVal:
                print(f"Table '{tableName}' has already been loaded for '{newCutoffDate}'.")
            else:

                process_table = p | f"Input Create Table for {tableName}" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])

                if len(table["modifyTable"]) > 0:
                    process_table = modify_table(table, process_table, last_cutoff_dates, [create_tables])

                return (
                    process_table
                    | f"Warehouse Table {tableName}" >> 
                        beam.ParDo(
                            WarehouseData(
                                upsert_config={
                                    "conn": destination_db["conn"],
                                    "database_type": destination_db["databaseType"],
                                    "database": destination_db["database"],
                                    "path": table["upsertTable"],
                                    "values_to_replace": [
                                        { "name": "NewCutoffDate", "value": newCutoffDate },
                                        { "name": "LastCutoffDate", "value": last_cutoff_date }
                                    ],
                                    "tables_to_replace": load_tables + warehouse_tables,
                                    "show_sql": False
                                },
                                assert_config={
                                    "conn": destination_db["conn"],
                                    "database_type": destination_db["databaseType"],
                                    "database": destination_db["database"],
                                    "test_config": config["testConfig"],
                                    "unit_tests": table["unitTests"],
                                    "values_to_replace": [
                                        { "name": "Database", "value": destination_db["database"] },
                                        { "name": "Schema", "value": table["schema"] },
                                        { "name": "Table", "value": table["table"] },
                                        { "name": "LoadDate", "value": newCutoffDate },
                                        { "name": "LastCutoffDate", "value": last_cutoff_date },
                                        { "name": "NewCutoffDate", "value": newCutoffDate }
                                    ],
                                    "tables_to_replace": load_tables + warehouse_tables,
                                    "test": config["test"],
                                    "test_record_count": config["testRecordCount"],
                                    "show_sql": False
                                },
                                insert_history_config={
                                    "conn": destination_db["conn"],
                                    "database_type": destination_db["databaseType"],
                                    "database": destination_db["database"],
                                    "path": config["warehouseHistory"]["insertWarehouseHistoryTable"],
                                    "values_to_replace": [
                                        { "name": "Database", "value": destination_db["database"] },
                                        { "name": "Schema", "value": config["warehouseHistory"]["schema"] },
                                        { "name": "Table", "value": config["warehouseHistory"]["table"] },
                                        { "name": "TableName", "value": tableName },
                                        { "name": "SchemaName", "value": table["schema"] },
                                        { "name": "LoadDate", "value": newCutoffDate },
                                        { "name": "LastCutoffDate", "value": last_cutoff_date },
                                        { "name": "Status", "value": "Successful" },
                                        { "name": "Details", "value": "" },
                                        { "name": "RollbackVersion", "value": str(table["rollbackVersion"]) }
                                    ],
                                    "tables_to_replace": [],
                                    "show_sql": False
                                },
                                yield_record={ f"Successfully processed table {tableName} for {newCutoffDate}." }
                            )
                        )
                    | f"Warehouse Table WaitOn {tableName}" >> WaitOn(*wait_on_process)
                    | f"Result for Table {tableName}" >> beam.Map(print)
                )
                
        # Create warehouse history table if not existing
        create_warehouse_history_table = (
            p
            | "Input Create Warehouse History Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
            | "Create Warehouse History Table" >> 
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        config["warehouseHistory"]["createWarehouseHistoryTable"],
                        [
                            { "name": "Database", "value": config["destination"]["database"] },
                            { "name": "Schema", "value": config["warehouseHistory"]["schema"] },
                            { "name": "Table", "value": config["warehouseHistory"]["table"] }
                        ],
                        [],
                        { "Successfully created warehouse history table." },
                        False,
                        destination_db["database"]
                    )
                )
            | "Print Create Warehouse History Table" >> beam.Map(print)
        )

        # Create warehouse history table if not existing
        create_warehouse_history_date_table = (
            p
            | "Input Create Warehouse History Date Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
            | "Create Warehouse History Date Table" >> 
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        config["warehouseHistoryDate"]["createWarehouseHistoryDateTable"],
                        [
                            { "name": "Database", "value": config["destination"]["database"] },
                            { "name": "Schema", "value": config["warehouseHistoryDate"]["schema"] },
                            { "name": "Table", "value": config["warehouseHistoryDate"]["table"] }
                        ],
                        [],
                        { "Successfully created warehouse history date table." },
                        False,
                        destination_db["database"]
                    )
                )
            | "Print Create Warehouse History Date Table" >> beam.Map(print)
        )

        # Create modify warehouse history table if not existing
        create_modify_warehouse_history_table = (
            p
            | "Input Create Modify Warehouse History Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
            | "Create Modify Warehouse History Table" >>
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        config["modifyWarehouseHistory"]["createModifyWarehouseHistoryTable"],
                        [
                            { "name": "Database", "value": config["destination"]["database"] },
                            { "name": "Schema", "value": config["modifyWarehouseHistory"]["schema"] },
                            { "name": "Table", "value": config["modifyWarehouseHistory"]["table"] }
                        ],
                        [],
                        { "Successfully created modify warehouse history table." },
                        False,
                        destination_db["database"]
                    )
                )
            | "Print Create Modify Warehouse History Table" >> beam.Map(print)
        )

        create_tables_sql = get_create_tables_sql(fact_tables + dimension_tables, last_cutoff_dates, newCutoffDate)
        create_tables = (
            p
            | "Input Create Tables" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
            | "Create Tables" >> beam.ParDo(
                CreateTables(
                    destination_db["databaseType"],
                    destination_db["conn"],
                    create_tables_sql,
                    destination_db["database"],
                    { f"Successfully created tables." },
                    False
                )
            )
            | "Wait On History Tables" >> WaitOn(create_warehouse_history_table, create_warehouse_history_date_table, create_modify_warehouse_history_table)
            | "Print Create Tables" >> beam.Map(print)
        )

        dimension_tables_processes = []
        if len(dimension_tables) > 0:
            for table in dimension_tables:    
                dimension_table_process = get_process_table(table, [create_tables], last_cutoff_dates)
                if dimension_table_process is not None:
                    dimension_tables_processes.append(dimension_table_process)

        fact_tables_processes = []    
        if len(fact_tables) > 0:
            if len(dimension_tables_processes) > 0:
                for table in fact_tables:
                    fact_table_process = get_process_table(table, dimension_tables_processes, last_cutoff_dates)
                    if fact_table_process is not None:
                        fact_tables_processes.append(fact_table_process)
            else:
                for table in fact_tables:
                    fact_table_process = get_process_table(table, [create_tables], last_cutoff_dates)
                    if fact_table_process is not None:
                        fact_tables_processes.append(fact_table_process)

        # Create warehouse history date
        # if len(dimension_tables_processes) > 0 or len(fact_tables_processes):
        (
            p 
            | "Input Insert Warehouse History Date Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
            | "Insert Warehouse History Date Table" >> 
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        config["warehouseHistoryDate"]["insertWarehouseHistoryDateTable"],
                        [
                            { "name": "Database", "value": config["destination"]["database"] },
                            { "name": "Schema", "value": config["warehouseHistoryDate"]["schema"] },
                            { "name": "Table", "value": config["warehouseHistoryDate"]["table"] },
                            { "name": "NewCutoffDate", "value": newCutoffDate },
                            { "name": "Status", "value": "Successful" },
                            { "name": "ArchivePath", "value": archivePath },
                            { "name": "SchemaWH", "value": config["warehouseHistory"]["schema"] },
                            { "name": "TableWH", "value": config["warehouseHistory"]["table"] },
                            { "name": "Tables", "value": ",".join(list(table["schema"] + "." + table["table"] for table in (config["dimensionTables"] + config["factTables"]))) },
                            { "name": "ReleaseGithubRepo", "value": release_github_repo },
                            { "name": "ReleaseGithubBranch", "value": release_github_branch },
                            { "name": "ReleaseGithubTag", "value": release_github_tag }
                        ],
                        [],
                        { "Successfully created Warehouse History table" },
                        False,
                        destination_db["database"] 
                    )
                )
            | f"WaitOn Insert Warehouse History Date" >> WaitOn(*(dimension_tables_processes + fact_tables_processes))
            | "Print Insert Warehouse History Date Table" >> beam.Map(print) 
        )

last_cutoff_dates: []
Processing table DimCities for 2013-01-01 00:00:00.
Processing table DimCustomers for 2013-01-01 00:00:00.
Processing table DimEmployees for 2013-01-01 00:00:00.
Processing table DimPaymentMethods for 2013-01-01 00:00:00.
Processing table DimStockItems for 2013-01-01 00:00:00.
Processing table DimSuppliers for 2013-01-01 00:00:00.
Processing table DimTransactionTypes for 2013-01-01 00:00:00.
Processing table DimDates for 2013-01-01 00:00:00.
Processing table FctMovements for 2013-01-01 00:00:00.
Processing table FctOrders for 2013-01-01 00:00:00.
Processing table FctPurchases for 2013-01-01 00:00:00.
Processing table FctSales for 2013-01-01 00:00:00.
Processing table FctStockHoldings for 2013-01-01 00:00:00.
Processing table FctTransactions for 2013-01-01 00:00:00.
{'Successfully created modify warehouse history table.'}
{'Successfully created warehouse history date table.'}
{'Successfully created warehouse history table.'}
{'Successfully created tables.'}
{'Succe

In [36]:
def get_warehouse_history_date_by_load_date(load_date):
    sql = sql_utils.get_sql_from_script(
        path=config["warehouseHistoryDate"]["loadWarehouseHistoryDateTable"], 
        values=[
            { "name": "Schema", "value": config["warehouseHistoryDate"]["schema"] },
            { "name": "Table", "value": config["warehouseHistoryDate"]["table"] },
            { "name": "Database", "value": destination_db["database"] },
            { "name": "LoadDate", "value": load_date }
        ], 
        tables=[],
        directory=script_directory
    )

    result = sql_utils.select_sql(
        destination_db["conn"], 
        sql, 
        destination_db["databaseType"], 
        "dictionary", 
        destination_db["database"], 
        spark_jars, 
        spark_master, 
        False, 
        False, 
        True
    )

    if len(result) > 0:
        return result[0]
    
    return None

def get_warehouse_histories_for_rollback(load_date):
    sql = sql_utils.get_sql_from_script(
        path=config["warehouseHistory"]["loadWarehouseHistoriesForRollback"], 
        values=[
            { "name": "Schema", "value": config["warehouseHistory"]["schema"] },
            { "name": "Table", "value": config["warehouseHistory"]["table"] },
            { "name": "Database", "value": destination_db["database"] },
            { "name": "LoadDate", "value": load_date }
        ], 
        tables=[],
        directory=script_directory
    )
    
    histories = sql_utils.select_sql(
        destination_db["conn"], 
        sql, 
        destination_db["databaseType"], 
        "dictionary", 
        destination_db["database"], 
        spark_jars, 
        spark_master, 
        False, 
        False, 
        True
    )

    histories_droprecords = list(history for history in histories if history["Type"] == "Drop Records")
    histories_non_droprecords = list(history for history in histories if history["Type"] != "Drop Records")

    history_sql = []
    for history in histories_droprecords:
        history_sql.append(
            sql_utils.get_sql_from_script(
                path=config["warehouseHistory"]["loadWarehouseHistoriesForRollbackDropRecords"], 
                values=[
                    { "name": "Schema", "value": history["SchemaName"] },
                    { "name": "Table", "value": history["TableName"] },
                    { "name": "Database", "value": destination_db["database"] },
                    { "name": "LoadDate", "value": load_date }
                ], 
                tables=[],
                database_type=destination_db["databaseType"],
                directory=script_directory
            )
        )

    if len(history_sql) > 0:
        history_sql = " UNION ALL ".join(history_sql)
        
        histories_droprecords_count = sql_utils.select_sql(
            destination_db["conn"], 
            history_sql, 
            destination_db["databaseType"], 
            "dictionary", 
            destination_db["database"], 
            spark_jars, 
            spark_master, 
            False, 
            False, 
            True
        )

        histories_droprecords_final = []
        for history in (history for history in histories_droprecords_count):
            orig_history = next((item for item in histories_droprecords if item.get("TableName") == history["TableName"] and item.get("SchemaName") == history["SchemaName"]), None)
            if history["CountRecreateFromInitialDate"] > 0:
                orig_history["Type"] = "Recreate Table"
            elif history["CountRecreateFromLastCutoffDate"] > 0:
                orig_history["Type"] = "Recreate Table From Last Cutoff Date"
            histories_droprecords_final.append(orig_history)
        histories_droprecords = histories_droprecords_final

    return histories_non_droprecords +  histories_droprecords

In [ ]:
if is_rollback == True:
    warehouse_history_date_data = get_warehouse_history_date_by_load_date(newCutoffDate)
    print(f"warehouse_history_date_data: {warehouse_history_date_data}")

    if warehouse_history_date_data is None:
        raise Exception(f"There is no warehouse history for the rollback date {newCutoffDate}.")

    warehouse_histories_for_rollback = get_warehouse_histories_for_rollback(newCutoffDate)
    print(f"warehouse_histories_for_rollback: {warehouse_histories_for_rollback}")

    options = PipelineOptions(runner_options)

    with beam.Pipeline(options=options) as p:
        tables = dimension_tables + fact_tables
        print(f"tables: {tables}")
        rollback_table_processes = []

        for tableHistory in warehouse_histories_for_rollback:
            rollback_type = tableHistory["Type"]
            tableName = tableHistory["TableName"]
            
            if rollback_type == "Load Date Is Latest":
                print(f"Table {tableName} latest load date is already {newCutoffDate}.")
            else:
                table = next(
                    (
                        table 
                        for table in tables 
                        if (
                            table["table"] == tableHistory["TableName"] and 
                            table["schema"] == tableHistory["SchemaName"]
                        )
                    ), 
                    None
                )
                
                added_columns = tableHistory["AddedColumns"]
                warehouse_last_cutoff_date = tableHistory["WarehouseLastCutoffDate"].strftime("%Y-%m-%d %H:%M:%S")
                
                print(f"Processing rollback of table '{tableName}' for '{newCutoffDate}'.")

                if table is not None:
                    if (rollback_type == "Recreate Table From Last Cutoff Date" or
                        rollback_type == "Recreate Table" or
                        rollback_type == "Drop Records"):
                        last_cutoff_date = initialCutoffDate
                        upsert_config = None
                        drop_table_config = None
                        recreate_table_config = None
                        
                        if rollback_type == "Recreate Table From Last Cutoff Date":
                            last_cutoff_date = warehouse_last_cutoff_date
                            upsert_config = {
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "path": table["upsertTable"],
                                "values_to_replace": [
                                    { "name": "NewCutoffDate", "value": newCutoffDate },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date }
                                ],
                                "tables_to_replace": load_tables + warehouse_tables,
                                "show_sql": False
                            }
                        elif rollback_type == "Recreate Table":
                            last_cutoff_date = initialCutoffDate
                            upsert_config = {
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "path": table["upsertTable"],
                                "values_to_replace": [
                                    { "name": "NewCutoffDate", "value": newCutoffDate },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date }
                                ],
                                "tables_to_replace": load_tables + warehouse_tables,
                                "show_sql": False
                            }
                            drop_table_config = {
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "path": table["dropTable"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["schema"] },
                                    { "name": "Table", "value": table["table"] },
                                ],
                                "tables_to_replace": [],
                                "show_sql": False
                            }
                            recreate_table_config = {
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "path": table["createTable"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["schema"] },
                                    { "name": "Table", "value": table["table"] }
                                ],
                                "tables_to_replace": [],
                                "show_sql": False
                            }
                        elif rollback_type == "Drop Records":
                            last_cutoff_date = warehouse_last_cutoff_date

                        rollback_table_processes.append(
                            (
                                p 
                                | f"Input Rollback Table {tableName}" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
                                | f"Rollback Table {tableName}" >> beam.ParDo(
                                        WarehouseData(
                                            rollback_droprecords_config={
                                                "conn": destination_db["conn"],
                                                "database_type": destination_db["databaseType"],
                                                "database": destination_db["database"],
                                                "path": config["rollBack"]["rollbackDropRecordsTable"],
                                                "drop_column_path": table["dropColumn"],
                                                "table": table["table"],
                                                "schema": table["schema"],
                                                "values_to_replace": [
                                                    { "name": "Database", "value": config["destination"]["database"] },
                                                    { "name": "Schema", "value": table["schema"] },
                                                    { "name": "Table", "value": table["table"] },
                                                    { "name": "LoadDate", "value": newCutoffDate },
                                                    { "name": "LastCutoffDate", "value": last_cutoff_date }
                                                ],
                                                "tables_to_replace": [],
                                                "set_timestamp_tostring": False,
                                                "show_sql": False,
                                                "added_columns": added_columns
                                            },
                                            upsert_config=upsert_config,
                                            assert_config={
                                                "conn": destination_db["conn"],
                                                "database_type": destination_db["databaseType"],
                                                "database": destination_db["database"],
                                                "test_config": config["testConfig"],
                                                "unit_tests": table["unitTests"],
                                                "values_to_replace": [
                                                    { "name": "Database", "value": destination_db["database"] },
                                                    { "name": "Schema", "value": table["schema"] },
                                                    { "name": "Table", "value": table["table"] },
                                                    { "name": "LoadDate", "value": newCutoffDate },
                                                    { "name": "LastCutoffDate", "value": last_cutoff_date },
                                                    { "name": "NewCutoffDate", "value": newCutoffDate }
                                                ],
                                                "tables_to_replace": load_tables + warehouse_tables,
                                                "test": config["test"],
                                                "test_record_count": config["testRecordCount"],
                                                "show_sql": False
                                            },
                                            rollback_drop_table_config=drop_table_config,
                                            rollback_recreate_table_config=recreate_table_config,
                                            rollback_final_config={
                                                "conn": destination_db["conn"],
                                                "database_type": destination_db["databaseType"],
                                                "database": destination_db["database"],
                                                "path": config["rollBack"]["rollbackDropRecordsFinalTable"],
                                                "values_to_replace": [
                                                    { "name": "LoadDate", "value": newCutoffDate },
                                                    { "name": "Database", "value": destination_db["database"] },
                                                    { "name": "Schema", "value": table["schema"] },
                                                    { "name": "Table", "value": table["table"] },
                                                    { "name": "SchemaWH", "value": config["warehouseHistory"]["schema"] },
                                                    { "name": "TableWH", "value": config["warehouseHistory"]["table"] },
                                                    { "name": "SchemaMWH", "value": config["modifyWarehouseHistory"]["schema"] },
                                                    { "name": "TableMWH", "value": config["modifyWarehouseHistory"]["table"] }
                                                ],
                                                "tables_to_replace": [],
                                                "show_sql": False
                                            },
                                            yield_record={ f"Successfully rolled back table {tableName} to {newCutoffDate}." }
                                        )
                                    )
                                | f"Print Rollback Table {tableName}" >> beam.Map(print)
                            )
                        )
                else:
                    if rollback_type == "Drop Table":
                        rollback_table_processes.append(
                            (
                                p 
                                | f"Input Rollback Table {tableName}" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
                                | f"Rollback Table {tableName}" >>
                                    beam.ParDo(
                                        UpsertData(
                                            destination_db["databaseType"],
                                            destination_db["conn"],
                                            config["rollBack"]["rollbackDropTable"],
                                            [
                                                { "name": "Database", "value": destination_db["database"] },
                                                { "name": "Schema", "value": tableHistory["SchemaName"] },
                                                { "name": "Table", "value": tableHistory["TableName"] },
                                                { "name": "SchemaWH", "value": config["warehouseHistory"]["schema"] },
                                                { "name": "TableWH", "value": config["warehouseHistory"]["table"] },
                                                { "name": "SchemaMWH", "value": config["modifyWarehouseHistory"]["schema"] },
                                                { "name": "TableMWH", "value": config["modifyWarehouseHistory"]["table"] }
                                            ],
                                            [],
                                            { "Successfully removed table" },
                                            False,
                                            destination_db["database"] 
                                        )
                                    )
                                | f"Rollback Final Table {tableName}" >>
                                    beam.ParDo(
                                        UpsertData(
                                            destination_db["databaseType"],
                                            destination_db["conn"],
                                            config["rollBack"]["rollbackDropFinalTable"],
                                            [
                                                { "name": "Database", "value": destination_db["database"] },
                                                { "name": "Schema", "value": tableHistory["SchemaName"] },
                                                { "name": "Table", "value": tableHistory["TableName"] },
                                                { "name": "SchemaWH", "value": config["warehouseHistory"]["schema"] },
                                                { "name": "TableWH", "value": config["warehouseHistory"]["table"] },
                                                { "name": "SchemaMWH", "value": config["modifyWarehouseHistory"]["schema"] },
                                                { "name": "TableMWH", "value": config["modifyWarehouseHistory"]["table"] },
                                                { "name": "LoadDate", "value": newCutoffDate },
                                            ],
                                            [],
                                            { "Successfully removed table" },
                                            False,
                                            destination_db["database"] 
                                        )
                                    )
                            )
                        )

        if len(rollback_table_processes) > 0:
            (
                p 
                | f"Input Rollback Warehouse History Date Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
                | f"Rollback Warehouse History Date Table" >>
                        beam.ParDo(
                            UpsertData(
                                destination_db["databaseType"],
                                destination_db["conn"],
                                config["warehouseHistoryDate"]["deleteWarehouseHistoryDatesForRollback"],
                                [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": config["warehouseHistoryDate"]["schema"] },
                                    { "name": "Table", "value": config["warehouseHistoryDate"]["table"] },
                                    { "name": "SchemaWH", "value": config["warehouseHistory"]["schema"] },
                                    { "name": "TableWH", "value": config["warehouseHistory"]["table"] },
                                    { "name": "LoadDate", "value": newCutoffDate }
                                ],
                                [],
                                { "Successfully rolled back Warehouse History Date" },
                                False,
                                destination_db["database"] 
                            )
                        )
                | f"WaitOn Rollback Load History Date" >> WaitOn(*rollback_table_processes)
                | "Print Rollback Load History Date" >> beam.Map(print)
            )

warehouse_history_date_data: {'WarehouseHistoryDateID': 1, 'LoadDate': datetime.datetime(2013, 1, 1, 0, 0), 'Status': 'Successful', 'ProcessedDate': datetime.datetime(2026, 6, 29, 5, 45, 8, 180000), 'ArchivePath': 'c:\\Docker\\WideWorldImportersELT_ApacheBeam\\notebooks\\warehouse'}
warehouse_histories_for_rollback: [{'TableName': 'DimCustomers', 'SchemaName': 'dbo', 'Type': 'Recreate Table', 'AddedColumns': '', 'WarehouseLastCutoffDate': datetime.datetime(2012, 12, 31, 0, 0)}, {'TableName': 'DimCountries', 'SchemaName': 'dbo', 'Type': 'Drop Table', 'AddedColumns': '', 'WarehouseLastCutoffDate': datetime.datetime(2012, 12, 31, 0, 0)}, {'TableName': 'DimStockItems', 'SchemaName': 'dbo', 'Type': 'Drop Records', 'AddedColumns': '', 'WarehouseLastCutoffDate': datetime.datetime(2012, 12, 31, 0, 0)}, {'TableName': 'DimSuppliers', 'SchemaName': 'dbo', 'Type': 'Drop Records', 'AddedColumns': '', 'WarehouseLastCutoffDate': datetime.datetime(2012, 12, 31, 0, 0)}, {'TableName': 'DimPaymentMethods